# CO3133 Assignment 2 — M3 Final (Val + Official VOC2007 Test Evaluation on A100)

Chạy toàn bộ từ trên xuống dưới (Run All) trên Google Colab **NVIDIA A100** (~80 phút).

In [1]:
# Cell 1: Unpack self-contained A2_M3 code package and optionally mount Google Drive for checkpoint backup
import base64, zipfile
from pathlib import Path

WORK_DIR = Path('/content/CO3133_A2_M3')
WORK_DIR.mkdir(parents=True, exist_ok=True)
STARTER_ZIP = WORK_DIR / 'CO3133_A2_M3_Starter.zip'

EMBEDDED_ZIP_B64 = (
    "..."
)

STARTER_ZIP.write_bytes(base64.b64decode(EMBEDDED_ZIP_B64))
with zipfile.ZipFile(STARTER_ZIP, 'r') as zf:
    zf.extractall(WORK_DIR)
print('Unpacked A2_M3 workspace to:', WORK_DIR)

SAVE_CHECKPOINTS_TO_DRIVE = True
if SAVE_CHECKPOINTS_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print('Google Drive mounted.')
    except Exception as e:
        print('Drive mount skipped:', e)


Unpacked A2_M3 workspace to: /content/CO3133_A2_M3
Mounted at /content/drive
Google Drive mounted.


In [2]:
# Cell 2: Verify NVIDIA A100 GPU and install dependencies
%cd /content/CO3133_A2_M3
!pip install -q iterative-stratification Markdown

import torch, torchvision
print('PyTorch:', torch.__version__, '| Torchvision:', torchvision.__version__)
assert torch.cuda.is_available(), 'CUDA is unavailable! Select Runtime -> Change runtime type -> A100 GPU.'
print('Assigned GPU:', torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')


/content/CO3133_A2_M3
PyTorch: 2.14.1+cu130 | Torchvision: 0.29.1+cu130
Assigned GPU: NVIDIA L4
VRAM: 22.03 GB


In [3]:
# Cell 3: Train 3 models (seed=42), verify Val best.pt, download VOCtest_06-Nov-2007.tar, check overlap, and evaluate Official Test Set once
!python run_a2.py --stage all_m3 --device cuda --out outputs/a2_m3_a100


{
  "group": "G-M10",
  "members": [
    "Trần Gia Lâm (2352670)",
    "Nguyễn Hữu Cầu (2352129)"
  ],
  "seed": 42,
  "dataset": "PASCAL VOC 2007",
  "archive_name": "VOCtrainval_06-Nov-2007.tar",
  "archive_url": "https://thor.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar",
  "archive_md5": "c52e279531787c972589f7e41ab4ae64",
  "test_archive_name": "VOCtest_06-Nov-2007.tar",
  "test_archive_url": "https://pjreddie.com/media/files/VOCtest_06-Nov-2007.tar",
  "test_archive_md5": "b6e924de25625d8de591ea690078ad9f",
  "num_classes": 21,
  "iou_threshold": 0.5,
  "score_threshold": 0.05,
  "nms_threshold": 0.5,
  "max_detections_per_image": 100,
  "pilot_batches": 200,
  "num_workers": 2,
  "cpu_threads": 4,
  "use_amp": true,
  "device": "cuda",
  "models": {
    "ssdlite320": {
      "architecture": "ssdlite320_mobilenet_v3_large",
      "init_weights": "MobileNet_V3_Large_Weights.IMAGENET1K_V1 backbone + fresh 21-class heads",
      "freeze_backbone": false,
      "min

In [4]:
# Cell 4: Backup best.pt checkpoints to Google Drive and inspect Val vs. Official Test results
import json, shutil
from pathlib import Path

out_dir = Path('/content/CO3133_A2_M3/outputs/a2_m3_a100')
print('=== VAL VS OFFICIAL TEST COMPARISON ===')
print((out_dir / 'test_comparison.md').read_text(encoding='utf-8'))

overlap = json.loads((out_dir / 'test_overlap_check.json').read_text(encoding='utf-8'))
print('=== TRAINVAL VS TEST OVERLAP CHECK ===')
print(json.dumps({k: overlap[k] for k in ['trainval_images', 'test_images', 'image_id_overlap_count', 'exact_sha256_overlap_count', 'near_duplicate_dhash_le_4_count']}, indent=2))

drive_ckpt_dir = Path('/content/drive/MyDrive/CO3133_A2_M3_Checkpoints')
if Path('/content/drive/MyDrive').exists():
    drive_ckpt_dir.mkdir(parents=True, exist_ok=True)
    for m in ['ssdlite320', 'fasterrcnn_frozen', 'fasterrcnn_finetune']:
        ckpt = out_dir / m / 'best.pt'
        if ckpt.exists():
            dst = drive_ckpt_dir / f'{m}_best.pt'
            shutil.copy2(ckpt, dst)
            print('Backed up checkpoint to Drive:', dst)


=== VAL VS OFFICIAL TEST COMPARISON ===
| Model | Selected Epoch | Val mAP@0.5 | Official Test mAP@0.5 | Delta (Test - Val) | Test Small AP | Test Med AP | Test Large AP |
|---|---:|---:|---:|---:|---:|---:|---:|
| `ssdlite320` | 17 | 50.70% | **52.98%** | +2.28 pp | 0.79% | 32.75% | 64.98% |
| `fasterrcnn_frozen` | 11 | 83.58% | **83.65%** | +0.07 pp | 45.32% | 75.15% | 81.89% |
| `fasterrcnn_finetune` | 12 | 75.51% | **76.90%** | +1.40 pp | 32.89% | 63.46% | 76.36% |

=== TRAINVAL VS TEST OVERLAP CHECK ===
{
  "trainval_images": 5011,
  "test_images": 4952,
  "image_id_overlap_count": 0,
  "exact_sha256_overlap_count": 0,
  "near_duplicate_dhash_le_4_count": 59
}
Backed up checkpoint to Drive: /content/drive/MyDrive/CO3133_A2_M3_Checkpoints/ssdlite320_best.pt
Backed up checkpoint to Drive: /content/drive/MyDrive/CO3133_A2_M3_Checkpoints/fasterrcnn_frozen_best.pt
Backed up checkpoint to Drive: /content/drive/MyDrive/CO3133_A2_M3_Checkpoints/fasterrcnn_finetune_best.pt


In [5]:
# Cell 5: Package all M3 outputs (JSONs, CSVs, PNGs) into CO3133_A2_M3_Outputs.zip and download
import zipfile
from pathlib import Path

out_root = Path('/content/CO3133_A2_M3/outputs/a2_m3_a100')
out_zip = Path('/content/CO3133_A2_M3_Outputs.zip')

with zipfile.ZipFile(out_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fp in sorted(out_root.rglob('*')):
        if fp.is_file() and fp.suffix != '.pt':
            arc = Path('outputs/a2_m3_a100') / fp.relative_to(out_root)
            zf.write(fp, arcname=arc.as_posix())

print(f'Created {out_zip} ({out_zip.stat().st_size / 1024:.1f} KB)')
try:
    from google.colab import files
    files.download(str(out_zip))
except Exception as e:
    print('Download manually from Colab Files pane:', out_zip)


Created /content/CO3133_A2_M3_Outputs.zip (23144.6 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>